# Load libraries

In [ ]:
import os
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import xgboost as xgb
from xgboost import XGBRegressor

import skopt
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold

# Custom functions

In [ ]:
def pickle_dump(path, saveobj):
    import pickle
    filehandler = open(path,"wb")
    pickle.dump(saveobj,filehandler)
    print("File pickled")
    filehandler.close()

In [ ]:
def pickle_load(path):
    import pickle
    file = open(path,'rb')
    loadobj = pickle.load(file)
    file.close()
    return loadobj

In [ ]:
def rmspe(y_true, y_pred):
    return  (np.sqrt(np.mean(np.square((y_true - y_pred) / y_true))))

In [ ]:
def feval_rmspe(y_pred, xgb_dtrain):
    y_true = xgb_dtrain.get_label()
    return "RMSPE", rmspe(y_true, y_pred)

# Read in data

In [ ]:
train_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/train.csv")
test_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/test.csv")
submit_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/sample_submission.csv")

opt_train_df = pd.read_feather("/kaggle/input/optiver-training-data/optiver_train3.feather")

In [ ]:
display(train_df.head(2))
display(test_df.head(2))
display(submit_df.head(2))

In [ ]:
opt_train_df.shape

# Clean up training data

In [ ]:
opt_train_df['time_num_seconds'] = opt_train_df['time_num_seconds'].apply(lambda x: len(x))
opt_train_df['time_num_seconds_norm'] = opt_train_df['time_num_seconds'] / 600
opt_train_df[['time_num_seconds','time_num_seconds_norm']].head()

In [ ]:
opt_train_df['time_dynamic_static_ratio'] = np.where(opt_train_df['time_OB_static_seconds_sum']!=0, opt_train_df['time_OB_total_dynamic_seconds'] / opt_train_df['time_OB_static_seconds_sum'], 0)

opt_train_df['OB_bid_price1_up_down_ratio'] = np.where(opt_train_df['OB_bid_price1_down_sum']!=0, opt_train_df['OB_bid_price1_up_sum'] / opt_train_df['OB_bid_price1_down_sum'], 0)

opt_train_df['OB_bid_price2_up_down_ratio'] = np.where(opt_train_df['OB_bid_price2_down_sum']!=0, opt_train_df['OB_bid_price2_up_sum'] / opt_train_df['OB_bid_price2_down_sum'], 0)

opt_train_df['OB_ask_price1_up_down_ratio'] = np.where(opt_train_df['OB_ask_price1_down_sum']!=0, opt_train_df['OB_ask_price1_up_sum'] / opt_train_df['OB_ask_price1_down_sum'], 0)

opt_train_df['OB_ask_price2_up_down_ratio'] = np.where(opt_train_df['OB_ask_price2_down_sum']!=0, opt_train_df['OB_ask_price2_up_sum'] / opt_train_df['OB_ask_price2_down_sum'], 0)

In [ ]:
opt_train_df.head()

In [ ]:
opt_train_df.replace([np.inf, -np.inf], np.nan, inplace=True)

opt_train_df.isnull().sum().sum()

In [ ]:
opt_train_df['real_vol_diff12'] = opt_train_df['real_vol1'] - opt_train_df['real_vol2']
opt_train_df['real_vol_diff13'] = opt_train_df['real_vol1'] - opt_train_df['real_vol3']
opt_train_df['real_vol_diff14'] = opt_train_df['real_vol1'] - opt_train_df['real_vol4']
opt_train_df['real_vol_diff23'] = opt_train_df['real_vol2'] - opt_train_df['real_vol3']
opt_train_df['real_vol_diff24'] = opt_train_df['real_vol2'] - opt_train_df['real_vol4']
opt_train_df['real_vol_diff34'] = opt_train_df['real_vol3'] - opt_train_df['real_vol4']

opt_train_df['wap1_range_dollar_change_abs_ratio'] = opt_train_df['wap1_range'] / opt_train_df['OB_wap1_dollar_change_abs_sum']
opt_train_df['wap2_range_dollar_change_abs_ratio'] = opt_train_df['wap2_range'] / opt_train_df['OB_wap2_dollar_change_abs_sum']
opt_train_df['wap3_range_dollar_change_abs_ratio'] = opt_train_df['wap3_range'] / opt_train_df['OB_wap3_dollar_change_abs_sum']
opt_train_df['wap4_range_dollar_change_abs_ratio'] = opt_train_df['wap4_range'] / opt_train_df['OB_wap4_dollar_change_abs_sum']

print(opt_train_df.shape)
opt_train_df.isnull().sum().sum()

In [ ]:
# opt_train_df['wap_range'] = opt_train_df['wap_min'] - opt_train_df['wap_max']
# opt_train_df['wap2_range'] = opt_train_df['wap2_min'] - opt_train_df['wap2_max']
# opt_train_df['bid_ask_diff_range'] = opt_train_df['bid_ask_diff_max'] - opt_train_df['bid_ask_diff_min']

# Data split

In [ ]:
opt_train_df['fold'] = -1

group_kfold = GroupKFold(n_splits=5)

kfold_val_dict = dict()

for fold, (_,val_idx) in enumerate(group_kfold.split(opt_train_df,opt_train_df['target'].values,opt_train_df['time_id'])):
    opt_train_df.loc[val_idx, 'fold'] = fold
    
    kfold_val_dict[fold] = val_idx

opt_train_df['fold'].value_counts()

# KFold training 1

In [ ]:
import lightgbm as lgb

In [ ]:
# Function to early stop with root mean squared percentage error
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))

def feval_rmspe(y_pred, lgb_train):
    y_true = lgb_train.get_label()
    return 'RMSPE', rmspe(y_true, y_pred), False

In [ ]:
seed0=2021
params = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.72,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
#     'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'device':'gpu',
    'verbose': -1}

In [ ]:
# StratifiedGroupKFold - Run training
oof_preds = np.zeros(opt_train_df.shape[0])

var_imp = pd.DataFrame()

for fold in tqdm(range(5)):

    print(f"Training fold {fold}")
    
    trn_x = opt_train_df[opt_train_df['fold']!=fold].drop(columns=['time_id','target','fold'])
    trn_y = opt_train_df[opt_train_df['fold']!=fold]['target'].values
    
    val_x =opt_train_df[opt_train_df['fold']==fold].drop(columns=['time_id','target','fold'])
    val_y = opt_train_df[opt_train_df['fold']==fold]['target'].values
    
    val_idx = kfold_val_dict[fold]
     
    # Root mean squared percentage error weights
    train_weights = 1 / np.square(trn_y)
    val_weights = 1 / np.square(val_y)
    train_dataset = lgb.Dataset(trn_x, trn_y, weight = train_weights)
    val_dataset = lgb.Dataset(val_x, val_y, weight = val_weights)
    model = lgb.train(params = params,
                      num_boost_round=10000,
                      train_set = train_dataset, 
                      valid_sets = [train_dataset, val_dataset], 
                      verbose_eval = 50,
                      early_stopping_rounds=50,
                      feval = feval_rmspe)
    
    
    var_imp[f'Fold{fold+1}'] = pd.Series(model.feature_importance(importance_type='gain'), index=trn_x.columns)
    
    oof_preds[val_idx] = model.predict(val_x)
    
    R2 = round(r2_score(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    RMSPE = round(rmspe(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    print(f'Fold {fold+1}: R2 score: {R2}, RMSPE: {RMSPE}')
    
    del model, trn_x, trn_y, val_x, val_y
    _ = gc.collect()

R2 = round(r2_score(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
RMSPE = round(rmspe(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
print(f'OOF: R2 score: {R2}, RMSPE: {RMSPE}')

# Feature importance - KFold 1

In [ ]:
varImp_df = pd.DataFrame(var_imp)
varImp_df['Avg'] = (varImp_df['Fold1'] + varImp_df['Fold2'] + varImp_df['Fold3'] + varImp_df['Fold4'] + varImp_df['Fold5']) / 5
varImp_df.shape

In [ ]:
varImp_df

In [ ]:
plt.hist(var_imp['Avg'].sort_values(ascending=False))

In [ ]:
plt.hist(var_imp['Avg'].sort_values(ascending=False)[var_imp['Avg'].sort_values(ascending=False)<10000])

In [ ]:
varImp_df['Avg'].nlargest(30).plot(kind='barh', figsize=(20,10))
plt.show()

In [ ]:
varImp_df['Avg'].sort_values(ascending=False)[varImp_df['Avg'].sort_values(ascending=False) > 1000]

In [ ]:
modelCols = varImp_df['Avg'].sort_values(ascending=False)[varImp_df['Avg'].sort_values(ascending=False) > 1000].index.tolist()
len(modelCols)

In [ ]:
modelCols

# Skopt training

In [ ]:
seed0=2021
SEARCH_PARAMS = {'learning_rate': 0.4,
                'max_depth': 4,
                'num_leaves': 32,
                'feature_fraction': 0.8,
                'subsample': 0.2,
                'lambda_l1': 0.5,
                'lambda_l2': 1.0,}



FIXED_PARAMS= {
                'objective': 'rmse',
                'boosting':'gbdt',
                'num_boost_round':10000,
                'early_stopping_rounds':50, 
                'min_data_in_leaf':500,
                'max_bin':300,
                'seed':seed0,
                'feature_fraction_seed': seed0,
                'bagging_seed': seed0,
                'drop_seed': seed0,
                'data_random_seed': seed0,
                'n_jobs':-1,
                'device':'gpu',
                'verbose': -1}

In [ ]:
def train_evaluate(search_params):
    
    fold=0
    
    X_train = opt_train_df[opt_train_df['fold']!=fold].drop(columns=['time_id','target','fold'])[modelCols]
    y_train = opt_train_df[opt_train_df['fold']!=fold]['target'].values
    
    X_valid =opt_train_df[opt_train_df['fold']==fold].drop(columns=['time_id','target','fold'])[modelCols]
    y_valid = opt_train_df[opt_train_df['fold']==fold]['target'].values
    
    # train_data = lgb.Dataset(X_train, label=y_train)
    # valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

    # Root mean squared percentage error weights
    train_weights = 1 / np.square(y_train)
    val_weights = 1 / np.square(y_valid)
    train_dataset = lgb.Dataset(X_train, y_train, weight = train_weights)
    val_dataset = lgb.Dataset(X_valid, y_valid, weight = val_weights)

    params = {
             'objective':FIXED_PARAMS['objective'],
             'min_data_in_leaf':FIXED_PARAMS['min_data_in_leaf'],
             'device':FIXED_PARAMS['device'],
             'boosting':FIXED_PARAMS['boosting'],
             'n_jobs':FIXED_PARAMS['n_jobs'],
             'verbose':FIXED_PARAMS['verbose'],
             **search_params}
    
    evals_result = {}

    model = lgb.train(params = params,
                      num_boost_round=FIXED_PARAMS['num_boost_round'],
                      train_set = train_dataset, 
                      valid_sets = [train_dataset, val_dataset], 
                      verbose_eval = 50, 
                      early_stopping_rounds=FIXED_PARAMS['early_stopping_rounds'],
                      valid_names=['train','valid'],
                      evals_result=evals_result,
                      feval = feval_rmspe)

    score = model.best_score['valid']['RMSPE']
    return score

In [ ]:
SPACE = [
    skopt.space.Real(0.001, 0.5, name='learning_rate', prior='log-uniform'),
    skopt.space.Integer(1, 10, name='max_depth'),
#     skopt.space.Integer(500, 5000, name='min_data_in_leaf'),
    skopt.space.Integer(30, 300, name='num_leaves'),
    skopt.space.Real(0.5, 1.0, name='feature_fraction', prior='uniform'),
    skopt.space.Real(0.4, 1.0, name='subsample', prior='uniform'),
    skopt.space.Real(0.001, 1000, name='lambda_l1', prior='log-uniform'),
    skopt.space.Real(0.001, 1000, name='lambda_l2', prior='log-uniform'),
]
@skopt.utils.use_named_args(SPACE)

def objective(**params):
    return train_evaluate(params)


results = skopt.forest_minimize(objective, SPACE, 
                                n_calls=100, n_initial_points=10, random_state = 101,
                                verbose=True
#                                 callback=[monitor]
                               )


In [ ]:
skopt_df = pd.DataFrame()

learning_rate = []
max_depth = []
num_leaves = []
feature_fraction = []
subsample = []
RMSPE = []

for params, score in zip(results.x_iters, results.func_vals):
    learning_rate.append(params[0])
    max_depth.append(params[1])
    num_leaves.append(params[2])
    feature_fraction.append(params[3])
    subsample.append(params[4])
    AUC.append(score)

skopt_df['learning_rate'] = learning_rate 
skopt_df['max_depth'] = max_depth 
skopt_df['num_leaves'] = num_leaves 
skopt_df['feature_fraction'] = feature_fraction 
skopt_df['subsample'] = subsample 
skopt_df['AUC'] = AUC 
skopt_df['AUC'] = skopt_df['AUC'] * -1

In [ ]:
skopt_df.sort_values(by=['AUC'], ascending=False)

# KFold training 2

In [ ]:
seed0=2021
params = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.7,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
#     'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'device':'gpu',
    'verbose': -1}

In [ ]:
# StratifiedGroupKFold - Run training
oof_preds = np.zeros(opt_train_df.shape[0])

var_imp = pd.DataFrame()

for fold in tqdm(range(5)):

    print(f"Training fold {fold}")
    
    trn_x = opt_train_df[opt_train_df['fold']!=fold].drop(columns=['time_id','target','fold'])[modelCols]
    trn_y = opt_train_df[opt_train_df['fold']!=fold]['target'].values
    
    val_x =opt_train_df[opt_train_df['fold']==fold].drop(columns=['time_id','target','fold'])[modelCols]
    val_y = opt_train_df[opt_train_df['fold']==fold]['target'].values
    
    val_idx = kfold_val_dict[fold]
     
    # Root mean squared percentage error weights
    train_weights = 1 / np.square(trn_y)
    val_weights = 1 / np.square(val_y)
    train_dataset = lgb.Dataset(trn_x, trn_y, weight = train_weights)
    val_dataset = lgb.Dataset(val_x, val_y, weight = val_weights)
    model = lgb.train(params = params,
                      num_boost_round=10000,
                      train_set = train_dataset, 
                      valid_sets = [train_dataset, val_dataset], 
                      verbose_eval = 50,
                      early_stopping_rounds=50,
                      feval = feval_rmspe)
    
    
    var_imp[f'Fold{fold+1}'] = pd.Series(model.feature_importance(), index=trn_x.columns)
    
    oof_preds[val_idx] = model.predict(val_x)
    
    R2 = round(r2_score(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    RMSPE = round(rmspe(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    print(f'Fold {fold+1}: R2 score: {R2}, RMSPE: {RMSPE}')
    
    del model, trn_x, trn_y, val_x, val_y
    _ = gc.collect()

R2 = round(r2_score(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
RMSPE = round(rmspe(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
print(f'OOF: R2 score: {R2}, RMSPE: {RMSPE}')

# Feature importance - KFold 2

In [ ]:
varImp_df = pd.DataFrame(var_imp)
varImp_df['Avg'] = (varImp_df['Fold1'] + varImp_df['Fold2'] + varImp_df['Fold3'] + varImp_df['Fold4'] + varImp_df['Fold5']) / 5
varImp_df.shape

In [ ]:
varImp_df

In [ ]:
plt.hist(var_imp['Avg'].sort_values(ascending=False))

In [ ]:
varImp_df['Avg'].nlargest(70).plot(kind='barh', figsize=(20,10))
plt.show()

In [ ]:
varImp_df['Avg'].sort_values(ascending=False)[varImp_df['Avg'].sort_values(ascending=False) > 400]

In [ ]:
modelCols2 = varImp_df['Avg'].sort_values(ascending=False)[varImp_df['Avg'].sort_values(ascending=False) > 400].index.tolist()
modelCols2

# KFold training 3

In [ ]:
seed0=2021
params = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.7,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
#     'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'device':'gpu',
    'verbose': -1}

In [ ]:
# StratifiedGroupKFold - Run training
oof_preds = np.zeros(opt_train_df.shape[0])

var_imp = pd.DataFrame()

for fold in tqdm(range(5)):

    print(f"Training fold {fold}")
    
    trn_x = opt_train_df[opt_train_df['fold']!=fold].drop(columns=['time_id','target','fold'])[modelCols2]
    trn_y = opt_train_df[opt_train_df['fold']!=fold]['target'].values
    
    val_x =opt_train_df[opt_train_df['fold']==fold].drop(columns=['time_id','target','fold'])[modelCols2]
    val_y = opt_train_df[opt_train_df['fold']==fold]['target'].values
    
    val_idx = kfold_val_dict[fold]
     
    # Root mean squared percentage error weights
    train_weights = 1 / np.square(trn_y)
    val_weights = 1 / np.square(val_y)
    train_dataset = lgb.Dataset(trn_x, trn_y, weight = train_weights)
    val_dataset = lgb.Dataset(val_x, val_y, weight = val_weights)
    model = lgb.train(params = params,
                      num_boost_round=10000,
                      train_set = train_dataset, 
                      valid_sets = [train_dataset, val_dataset], 
                      verbose_eval = 50,
                      early_stopping_rounds=50,
                      feval = feval_rmspe)
    
    
    var_imp[f'Fold{fold+1}'] = pd.Series(model.feature_importance(), index=trn_x.columns)
    
    oof_preds[val_idx] = model.predict(val_x)
    
    R2 = round(r2_score(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    RMSPE = round(rmspe(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    print(f'Fold {fold+1}: R2 score: {R2}, RMSPE: {RMSPE}')
    
    del model, trn_x, trn_y, val_x, val_y
    _ = gc.collect()

R2 = round(r2_score(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
RMSPE = round(rmspe(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
print(f'OOF: R2 score: {R2}, RMSPE: {RMSPE}')

# Final training

In [ ]:
modelCols

In [ ]:
modelCols = ['stock_id',
            'real_vol1',
            'ask_spread_max',
            'bid_spread_min',
            'bid_ask_spread_min',
            'real_vol2',
            'OB_wap2_dollar_change_abs_sum',
            'wapbal_sum',
            'realvol_diff3',
            'total_volume_min',
            'spread_std',
            'real_vol3',
            'OB_wap1_dollar_change_abs_sum',
            'OB_wap1_num_cross_mean',
            'realvol_diff2',
            'price_spread_std',
            'OB_bid_price2_up_down_ratio',
            'time_OB_static_seconds_max',
            'bid_spread_std',
            'ask_spread_sum',
            'OB_bid_price1_num_cross_mean',
            'time_OB_dynamic_seconds_max',
            'bid_ask_diff_wap2_ratio_mean',
            'bid_ask_diff_sum',
            'OB_wap1_same_change_ratio',
            'bid_spread_sum',
            'OB_bid_size2_dollar_change_abs_sum',
            'price_spread_sum',
            'real_vol4',
            'OB_ask_size2_dollar_change_abs_sum',
            'bid_ask_diff_wap_ratio_mean',
            'OB_wap3_dollar_change_abs_sum',
            'OB_wap3_same_sum',
            'OB_bid_price1_dollar_change_abs_sum',
            'realvol_diff1',
            'OB_ask_price1_dollar_change_abs_sum',
            'OB_bid_size1_same_sum',
            'bid_ask_spread_std',
            'ask_spread_std',
            'time_OB_static_seconds_sum',
            'bid_ask_spread_sum',
            'spread_sum',
            'time_OB_total_dynamic_groups',
            'OB_wap1_same_sum',
            'bid_ask_diff2_wap2_ratio_mean',
            'OB_ask_size1_up_sum',
            'OB_wap2_same_change_ratio',
            'OB_ask_price2_up_down_ratio',
            'price_spread2_std',
            'OB_ask_size1_same_sum',
            'OB_ask_price1_same_change_ratio']

In [ ]:
seed0=2021
params = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.72,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
#     'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'device':'gpu',
    'verbose': -1}

In [ ]:
trn_x = opt_train_df[modelCols]
trn_y = opt_train_df['target'].values

# Root mean squared percentage error weights
train_weights = 1 / np.square(trn_y)
train_dataset = lgb.Dataset(trn_x, trn_y, weight = train_weights)

model = lgb.train(params = params,
                  num_boost_round=550,
                  train_set = train_dataset, 
                  valid_sets = [train_dataset], 
                  verbose_eval = 50,
                  feval = feval_rmspe)

In [ ]:
pd.Series(model.feature_importance(), index=trn_x.columns)

In [ ]:
preds = model.predict(trn_x)

R2 = round(r2_score(y_true = trn_y, y_pred = preds),3)
RMSPE = round(rmspe(y_true = trn_y, y_pred = preds),3)
print(f'Train: R2 score: {R2}, RMSPE: {RMSPE}')

In [ ]:
pickle_dump("./lgbm_train3_med.pkl", model)